# SurgBot 分步演示

**五一 MVP 版本** — 无需机械臂，完整展示从语音指令到路径规划的全链路。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zebedee2021/surgbot-docs-v2/blob/main/notebooks/surgbot_demo.ipynb)

> 本页面由 GitHub Actions 自动构建，所有输出均为真实运行结果。
> 点击上方按钮可在 Colab 中交互运行。

In [ ]:
# ── 路径初始化（适配 CI 和本地两种环境）──────────────────
import sys, os
from pathlib import Path

# 找到仓库根目录（无论从哪里运行）
HERE = Path(os.getcwd())
for p in [HERE, HERE.parent, HERE.parent.parent]:
    candidate = p / 'surgbot'
    if candidate.exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        REPO_ROOT = p
        break

print(f'surgbot 代码路径: {candidate}')
print(f'Python {sys.version.split()[0]}')

## Step 1 — 配置加载

In [ ]:
from core.config import cfg

rows = [
    ('机械臂 IP',      cfg.robot.ip),
    ('运动速度',        f'{cfg.robot.speed} %'),
    ('抬升安全高度',    f'{cfg.robot.z_approach_offset} mm'),
    ('X 工作范围',     f'[{cfg.safety.x_min:.0f}, {cfg.safety.x_max:.0f}] mm'),
    ('Y 工作范围',     f'[{cfg.safety.y_min:.0f}, {cfg.safety.y_max:.0f}] mm'),
    ('Z 工作范围',     f'[{cfg.safety.z_min:.0f}, {cfg.safety.z_max:.0f}] mm'),
    ('力反馈阈值',      f'{cfg.safety.force_threshold} N'),
    ('单步最大距离',    f'{cfg.safety.max_single_step_dist} mm'),
]

print(f'{"参数":<16} {"值"}')
print('-' * 40)
for k, v in rows:
    print(f'{k:<16} {v}')

## Step 2 — 安全校验

In [ ]:
from core.safety_manager import safety, WorkspaceViolation, LargeStepDetected, SafetyError

CASES = [
    ('slot_01 夹取点',      [120.0, -80.0, 180.0, -180, 0, 45, 0], True),
    ('slot_04 夹取点',      [360.0, -80.0, 180.0, -180, 0,  0, 0], True),
    ('X 越界 (x=700)',      [700.0, -80.0, 180.0, -180, 0,  0, 0], False),
    ('Z 过低 (z=50, 撞盘)', [200.0, -80.0,  50.0, -180, 0,  0, 0], False),
    ('关节角模式（跳过）',   [0.0, 32.6, -129.1, 6.7, 90.0, -90.0, 1], True),
]

print(f'{"测试用例":<22} {"预期":<6} {"结果"}')
print('-' * 50)
all_ok = True
for label, pt, should_pass in CASES:
    try:
        safety.validate_point(pt, label=label)
        result = '✅ 通过' if should_pass else '❌ 应拦截'
        if not should_pass: all_ok = False
    except SafetyError:
        result = '🚫 已拦截' if not should_pass else '❌ 误拦截'
        if should_pass: all_ok = False
    print(f'{label:<22} {"合法" if should_pass else "违规":<6} {result}')

# 大步长检测
try:
    safety.validate_path([[100.0,-80.0,180.0,-180,0,0,0],
                           [100.0,-80.0,620.0,-180,0,0,0]])  # 步长 440mm
    print(f'{"大步长 440mm":<22} {"违规":<6} ❌ 应拦截')
    all_ok = False
except LargeStepDetected:
    print(f'{"大步长 440mm":<22} {"违规":<6} 🚫 已拦截')

print()
print('整体校验：', '✅ 全部符合预期' if all_ok else '❌ 有测试不符预期')

## Step 3 — 工作空间 & 器械槽位可视化

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

SLOTS = [
    {'id':'slot_01','name':'持针器_大','pt':[120.0,-80.0,180.0],'rz':45.0,'color':'#e74c3c'},
    {'id':'slot_02','name':'剪刀',     'pt':[200.0,-80.0,180.0],'rz': 0.0,'color':'#3498db'},
    {'id':'slot_03','name':'镊子',     'pt':[280.0,-80.0,180.0],'rz':90.0,'color':'#2ecc71'},
    {'id':'slot_04','name':'刀柄',     'pt':[360.0,-80.0,180.0],'rz':30.0,'color':'#f39c12'},
    {'id':'slot_05','name':'持针器_小','pt':[440.0,-80.0,180.0],'rz':45.0,'color':'#9b59b6'},
]

s = cfg.safety
xs=[s.x_min,s.x_max]; ys=[s.y_min,s.y_max]; zs=[s.z_min,s.z_max]
ex,ey,ez=[],[],[]
corners=[(x,y,z) for x in xs for y in ys for z in zs]
for i,(x1,y1,z1) in enumerate(corners):
    for x2,y2,z2 in corners[i+1:]:
        if sum([x1!=x2,y1!=y2,z1!=z2])==1:
            ex+=[x1,x2,None]; ey+=[y1,y2,None]; ez+=[z1,z2,None]

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=ex,y=ey,z=ez,mode='lines',
    line=dict(color='rgba(100,100,255,0.25)',width=2),name='工作空间边界',hoverinfo='skip'))

# 托盘
fig.add_trace(go.Scatter3d(
    x=[50,510,510,50,50],y=[-130,-130,-30,-30,-130],z=[178]*5,
    mode='lines',line=dict(color='#95a5a6',width=3),name='器械托盘'))

# 机械臂底座 & 递送点
fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers+text',
    marker=dict(size=12,color='#2c3e50',symbol='square'),
    text=['底座'],textposition='top center',name='机械臂底座'))
fig.add_trace(go.Scatter3d(x=[-50],y=[-250],z=[350],mode='markers+text',
    marker=dict(size=10,color='#1abc9c',symbol='diamond'),
    text=['递送点'],textposition='top center',name='递送点（医生侧）'))

# 槽位
for sl in SLOTS:
    p=sl['pt']; za=p[2]+cfg.robot.z_approach_offset
    fig.add_trace(go.Scatter3d(x=[p[0]],y=[p[1]],z=[p[2]],mode='markers+text',
        marker=dict(size=11,color=sl['color'],line=dict(width=2,color='white')),
        text=[sl['name']],textposition='top center',name=sl['name'],
        hovertemplate=f"<b>{sl['name']}</b><br>{sl['id']}<br>"
            f"({p[0]:.0f},{p[1]:.0f},{p[2]:.0f}) mm<br>Rz={sl['rz']}°"))
    fig.add_trace(go.Scatter3d(x=[p[0]],y=[p[1]],z=[za],mode='markers',
        marker=dict(size=5,color=sl['color'],symbol='circle-open'),
        showlegend=False,hovertemplate=f'approach Z={za:.0f}mm'))
    fig.add_trace(go.Scatter3d(x=[p[0],p[0]],y=[p[1],p[1]],z=[p[2],za],
        mode='lines',line=dict(color=sl['color'],width=1,dash='dot'),
        showlegend=False,hoverinfo='skip'))

fig.update_layout(
    title='工作空间 & 器械槽位分布',
    scene=dict(
        xaxis=dict(title='X (mm)'),yaxis=dict(title='Y (mm)'),zaxis=dict(title='Z (mm)'),
        aspectmode='manual',aspectratio=dict(x=1.3,y=1.4,z=1.0),
        camera=dict(eye=dict(x=1.5,y=-1.8,z=1.2))),
    margin=dict(l=0,r=0,b=0,t=40),height=560)
fig.show()

## Step 4 — 运动路径轨迹

In [ ]:
TARGET = 'slot_02'   # 剪刀
sl = next(s for s in SLOTS if s['id']==TARGET)
p = sl['pt']; za = p[2]+cfg.robot.z_approach_offset

DELIVER = [-50.0, -250.0, 350.0]
RESET   = [  0.0,  -50.0, 400.0]

phases = [
    ('待机',    RESET,             '#95a5a6'),
    ('approach',[p[0],p[1],za],    '#3498db'),
    ('grasp',   p,                 '#e74c3c'),
    ('lift',    [p[0],p[1],za],    '#e67e22'),
    ('deliver', DELIVER,           '#2ecc71'),
    ('reset',   RESET,             '#95a5a6'),
]

fig2 = go.Figure()
xs=[w[1][0] for w in phases]; ys=[w[1][1] for w in phases]; zs=[w[1][2] for w in phases]
fig2.add_trace(go.Scatter3d(x=xs,y=ys,z=zs,mode='lines',
    line=dict(color='#bdc3c7',width=3),name='路径总览',showlegend=False))

LABELS = {'待机':'⚪ 待机','approach':'🔵 approach','grasp':'🔴 grasp（夹取）',
          'lift':'🟠 lift','deliver':'🟢 deliver（递送）','reset':'⚪ reset'}
for name,pt,color in phases:
    fig2.add_trace(go.Scatter3d(x=[pt[0]],y=[pt[1]],z=[pt[2]],
        mode='markers+text',
        marker=dict(size=12,color=color,line=dict(width=2,color='white')),
        text=[LABELS[name]],textposition='top center',name=LABELS[name],
        hovertemplate=f'<b>{name}</b><br>({pt[0]:.0f},{pt[1]:.0f},{pt[2]:.0f}) mm'))

fig2.add_trace(go.Scatter3d(x=[p[0]],y=[p[1]],z=[p[2]],mode='markers',
    marker=dict(size=15,color=sl['color'],symbol='diamond',
                line=dict(width=2,color='white')),
    name=f'器械: {sl["name"]}'))

fig2.update_layout(
    title=f'运动轨迹 — {sl["name"]} ({TARGET})',
    scene=dict(
        xaxis=dict(title='X (mm)'),yaxis=dict(title='Y (mm)'),zaxis=dict(title='Z (mm)'),
        aspectmode='manual',aspectratio=dict(x=1.3,y=1.4,z=1.0),
        camera=dict(eye=dict(x=1.6,y=-1.6,z=1.2))),
    margin=dict(l=0,r=0,b=0,t=40),height=580)
fig2.show()

print(f'器械: {sl["name"]}  |  夹取 ({p[0]:.0f},{p[1]:.0f},{p[2]:.0f}) mm  '
      f'|  approach Z={za:.0f} mm  |  Rz={sl["rz"]}°')

## Step 5 — 完整 Mock 流程

In [ ]:
from core.interfaces import InstrumentCommand, GraspTarget
from hardware.dobot_arm import DobotArm
import time

# ① NLP 输出
cmd = InstrumentCommand(
    instrument_id='INS-031', name='持针器_大',
    confidence=0.93, source_text='递持针器', slot_id='slot_01')

# ② 感知输出
sl0 = SLOTS[0]
grasp = GraspTarget(
    slot_id='slot_01', instrument_id='INS-031',
    grasp_point=sl0['pt'], orientation_deg=sl0['rz'],
    confidence=0.96, visual_offset=[2.3,-1.1,0.0])

# ③ 安全校验
safety.validate_grasp(grasp)
print('安全校验 ✅')

# ④ 执行（mock，缩短 handover 超时到 2s 用于演示）
arm = DobotArm(mock=True)
arm._FORCE_WAIT_TIMEOUT = 2   # 演示用：2s 超时（真机由力反馈触发）

t0 = time.time()
arm.approach(grasp.grasp_point, grasp.orientation_deg)
gripped = arm.grasp(grasp.grasp_point, grasp.orientation_deg, gripper_preset_id=3)
arm.lift(grasp.grasp_point, grasp.orientation_deg)
arm.deliver()
taken = arm.wait_for_handover(timeout=2, gripper_preset_id=-1)
arm.reset()
elapsed = time.time()-t0
arm.shutdown()

print()
print('执行结果')
print('-'*40)
print(f'  器械      : {cmd.name}')
print(f'  夹取      : {"✅ 成功" if gripped else "❌ 失败"}')
print(f'  交接      : {"✅ 已取走" if taken else "⏱ 超时（真机由医生触发）"}')
print(f'  耗时      : {elapsed:.1f}s')
print(f'  急停次数  : {safety.stop_count}')

---

## 数据流总结

```
语音指令 → InstrumentCommand(confidence=0.93)
    → 槽位注册表查询 → GraspTarget(confidence=0.96)
    → safety.validate_grasp() + validate_path()  ✅
    → DobotArm.pick_and_deliver()
        approach(120, -80, 330) → grasp(120, -80, 181)
        → lift → deliver → wait_force → reset
```

完整交互版本：[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zebedee2021/surgbot-docs-v2/blob/main/notebooks/surgbot_demo.ipynb)